In [ ]:
from model import SchemaIntegrationRuns
from evaluation import SchemaIntegrationEvaluation
import os
from dotenv import dotenv_values
from langchain.chat_models import init_chat_model
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [ ]:
# Initialize LLM
model_name = "gpt-5.2-2025-12-11"
# model_name = "Qwen/Qwen3.6-27B"
reasoning = "None"
model_type = "openai"
# model_type = "hf" # For models loaded through HuggingFace

if model_type == "hf":
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype="float16"
    )
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir="hf_cache/")
    model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float16, low_cpu_mem_usage=True, device_map="auto", cache_dir="hf_cache/", quantization_config=quant_config)
    if "qwen" in model_name.lower():
        # Update chat format to fix issues with tool calls
        with open("preprocessing/qwen_chat_template.jinja", "r") as f:
            tokenizer.chat_template = f.read()
else:
    config = dotenv_values("../key.env")
    os.environ['OPENAI_API_KEY'] = config["OPENAI_API_KEY"]
    OPENAI_API_KEY = config["OPENAI_API_KEY"]
    
    if reasoning != "None":
        model = init_chat_model(openai_api_key=OPENAI_API_KEY, temperature=1, model=model_name, reasoning = {"effort": reasoning, "summary": "auto"})
    else:
        model = init_chat_model(openai_api_key=OPENAI_API_KEY, temperature=1, model=model_name)
    tokenizer = None

### Run all workflows

In [ ]:
prompt_name = "long-prompt-updated-4"
benchmark = "SINT-Benchmark"
for folder_name in os.listdir(f"data/selected-tables/{benchmark}/"):
    for sequence_of_phases in [
        ["detect_tables_phase", "schema_matching_phase", "grouping_phase", "schema_integration_phase", "final_integration_phase"],
        # ["schema_matching_phase", "schema_integration_phase", "detect_tables_phase", "final_integration_phase"],
        # ["detect_tables_phase", "grouping_phase", "schema_integration_phase", "final_integration_phase"]

    ]:
        # Run config
        config = {
            "folder_name": folder_name,
            "prompt_name": prompt_name,
            "table_path": "data/selected-tables/",
            "table_format": "markdown",
            "table_rows": 10,
            "model": model,
            "model_type": model_type,
            "model_name": model_name,
            "reasoning": reasoning,
            "tokenizer": tokenizer,
            "demonstration": 0,
            "self_consistency": True,
            "schema_matching_batch_size": 1,
            "num_runs": 3,
            "sequence_of_phases": sequence_of_phases,
        }
        schema_integration_workflow = SchemaIntegrationRuns(**config)
        schema_integration_workflow.run_workflow()

### Evaluation

In [ ]:
for log_file_name in os.listdir("logs/"):
    if f"{log_file_name.split('.json')[0]}_evaluation.json" not in os.listdir("evaluation_json/"):
        try:
            schema_evaluator = SchemaIntegrationEvaluation(tables_path="data/selected-tables/SINT-Benchmark", log_file_name=f"logs/{log_file_name.split('.json')[0]}")
            schema_evaluator.run_evaluation()
            schema_evaluator.save_results()
        except Exception as e:
            print(f"Error processing {log_file_name}: {e}")

### Manual workflow runs

In [ ]:
prompt_name = "long-prompt-updated-4"
folder_name = "goby_selected" # choose use-case
config = {
    "folder_name": folder_name,
    "prompt_name": prompt_name,
    "table_path": "data/selected-tables/",
    "table_format": "markdown",
    "table_rows": 10,
    "model": model,
    "model_type": model_type,
    "model_name": model_name,
    "reasoning": reasoning,
    "tokenizer": tokenizer,
    "demonstration": 0,
    "self_consistency": True,
    "schema_matching_batch_size": 1,
    "num_runs": 3,
    "sequence_of_phases": sequence_of_phases,
}

In [ ]:
schema_integration_workflow = SchemaIntegrationRuns(**config)
schema_integration_workflow.initialize_workflow()

In [ ]:
phase_name = "detect_tables_phase"
phase_function = schema_integration_workflow.get_function(phase_name=phase_name, run_nr=0)["function"]
phase_function(run_feedback_loop=True, fds=schema_integration_workflow.fds, perfect_case=False)

# Postprocessing
phase_postprocessing_function = schema_integration_workflow.get_function(phase_name=phase_name, run_nr=0)["postprocessing_function"]
if phase_postprocessing_function:
    phase_postprocessing_function()

In [ ]:
phase_name = "schema_matching_phase"
phase_function = schema_integration_workflow.get_function(phase_name=phase_name, run_nr=0)["function"]
phase_function(run_feedback_loop=True, fds=schema_integration_workflow.fds, schema_matching_batch_size=3)

# Postprocessing
phase_postprocessing_function = schema_integration_workflow.get_function(phase_name=phase_name, run_nr=0)["postprocessing_function"]
if phase_postprocessing_function:
    phase_postprocessing_function()

In [ ]:
phase_name = "grouping_phase"
phase_function = schema_integration_workflow.get_function(phase_name=phase_name, run_nr=0)["function"]
phase_function(run_feedback_loop=True, fds=schema_integration_workflow.fds, perfect_case=False)

# Postprocessing
phase_postprocessing_function = schema_integration_workflow.get_function(phase_name=phase_name, run_nr=0)["postprocessing_function"]
if phase_postprocessing_function:
    phase_postprocessing_function()

In [ ]:
phase_name = "schema_integration_phase"
phase_function = schema_integration_workflow.get_function(phase_name=phase_name, run_nr=0)["function"]
phase_function(run_feedback_loop=True, fds=schema_integration_workflow.fds, perfect_case=False, merging_batch_size="all")
# Postprocessing
phase_postprocessing_function = schema_integration_workflow.get_function(phase_name=phase_name, run_nr=0)["postprocessing_function"]
if phase_postprocessing_function:
    phase_postprocessing_function()

In [ ]:
phase_name = "final_integration_phase"
phase_function = schema_integration_workflow.get_function(phase_name=phase_name, run_nr=0)["function"]
phase_function(run_feedback_loop=True, fds=schema_integration_workflow.fds, perfect_case=False)

# Postprocessing
phase_postprocessing_function = schema_integration_workflow.get_function(phase_name=phase_name, run_nr=0)["postprocessing_function"]
if phase_postprocessing_function:
    phase_postprocessing_function()